In [14]:
LINK = "https://https://getintoyc.com/"

In [16]:
import selenium
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import time

# --- Configuration ---
# CORRECT THE LINK HERE!
LINK = "https://getintoyc.com/" # Make sure it's like this, not https://https://...

# Text to wait for (signals dynamic content is loaded)
WAIT_TEXT = "Code For Cash" 
# XPath to find any element containing the specific text
WAIT_ELEMENT_XPATH = f"//*[contains(text(), '{WAIT_TEXT}')]" 

WAIT_TIMEOUT = 30 # Timeout for waiting
OUTPUT_FILENAME = "yc_applications.html"
# --- End Configuration ---

driver = None # Initialize driver to None for finally block
try:
    print("Initializing Chrome driver...")
    driver = webdriver.Chrome() 
    
    print(f"Navigating to: {LINK}")
    driver.get(LINK) # This line was failing due to the bad URL

    # --- Wait for the specific text to appear ---
    print(f"Waiting up to {WAIT_TIMEOUT} seconds for text '{WAIT_TEXT}' to appear...")
    wait_condition = EC.visibility_of_element_located((By.XPATH, WAIT_ELEMENT_XPATH))
    WebDriverWait(driver, WAIT_TIMEOUT).until(wait_condition)
    print(f"Text '{WAIT_TEXT}' found. Dynamic content likely loaded.") 
    
    # Optional: Add a small, final sleep if needed for final rendering tweaks
    # print("Adding small sleep for final rendering...")
    # time.sleep(1) 

    # --- Extract HTML ---
    print("Extracting page source...")
    html = driver.page_source
    print(f"HTML length: {len(html)}")

    # --- Save HTML ---
    print(f"Saving HTML to {OUTPUT_FILENAME}...")
    with open(OUTPUT_FILENAME, "w", encoding="utf-8") as file:
        file.write(html)
    print("HTML saved.")

except Exception as e:
    if "TimeoutException" in str(type(e)):
        print(f"Timeout: Waited {WAIT_TIMEOUT} seconds, but text '{WAIT_TEXT}' did not become visible.")
    elif "net::ERR_NAME_NOT_RESOLVED" in str(e):
         print(f"Error: Could not resolve the hostname in the URL: {LINK}. Please check the URL.")
    else:
        print(f"An error occurred: {e}")

finally:
    # --- Close Browser ---
    if driver:
        print("Closing browser...")
        driver.quit()
    print("Script finished.")


Initializing Chrome driver...
Navigating to: https://getintoyc.com/
Waiting up to 30 seconds for text 'Code For Cash' to appear...
Text 'Code For Cash' found. Dynamic content likely loaded.
Extracting page source...
HTML length: 221868
Saving HTML to yc_applications.html...
HTML saved.
Closing browser...
Script finished.


In [21]:
# for each class="ct-link brand-color company" in the HTML, print the text and the link

import bs4

soup = bs4.BeautifulSoup(html, "html.parser")

links = []
for link in soup.find_all("a", class_="ct-link brand-color company"):
    link = link["href"]
    links.append(link)

links

['https://getintoyc.com/company/dropbox/',
 'https://getintoyc.com/company/gitlab/',
 'https://stackable.so/?utm_source=getintoychome',
 'https://getintoyc.com/company/buffer/',
 'https://getintoyc.com/company/paystack/',
 'https://getintoyc.com/company/mixpanel/',
 'https://getintoyc.com/company/openphone/',
 'https://getintoyc.com/company/seeing-interactive-own-local/',
 'https://getintoyc.com/company/simple-habit/',
 'https://getintoyc.com/company/flex/',
 'https://getintoyc.com/company/inevent/',
 'https://getintoyc.com/company/task-pigeon/',
 'https://getintoyc.com/company/mimir/',
 'https://getintoyc.com/company/cruise/',
 'https://getintoyc.com/company/virtually/',
 'https://getintoyc.com/company/dendron/',
 'https://getintoyc.com/company/learn-venue/',
 'https://getintoyc.com/company/laitum/',
 'https://getintoyc.com/company/streamplate/',
 'https://getintoyc.com/company/sketchdeck/',
 'https://getintoyc.com/company/slite/',
 'https://getintoyc.com/company/prolific/',
 'https:/

In [30]:
import json
import os

all_company_data = [] # Initialize list to store data for all companies
output_dir = "companies_htmls"
processed_data_dir = "data_processed" # Directory for the final JSON
    
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
if not os.path.exists(processed_data_dir):
    os.makedirs(processed_data_dir)

for link in links:
    company = link.strip('/').split("/")[-1] # Safer way to get last part
    if not company: # Handle potential trailing slash
         company = link.strip('/').split("/")[-2]
         
    driver = None # Initialize driver to None for finally block
    try:
        print(f"Processing: {company} ({link})")
        driver = webdriver.Chrome()
        driver.get(link)

        # --- Corrected Wait Condition ---
        # Use XPath to find any element containing the text "Batch"
        wait_text = "Batch" 
        wait_xpath = f"//*[contains(text(), '{wait_text}')]"
        print(f"Waiting for text '{wait_text}' using XPath: {wait_xpath}")
        
        # Wait for the element to be present or visible
        wait_condition = EC.presence_of_element_located((By.XPATH, wait_xpath))
        # or EC.visibility_of_element_located((By.XPATH, wait_xpath))
        
        WebDriverWait(driver, 20).until(wait_condition) # Increased timeout slightly
        print(f"Text '{wait_text}' found.")
        # --- End Corrected Wait ---

        print("Extracting page source...")
        html = driver.page_source
        
        output_path = os.path.join(output_dir, f"{company}.html")
        print(f"Saving HTML to: {output_path}")
        with open(output_path, "w", encoding="utf-8") as file:
            file.write(html)    
        
    except Exception as e:
        print(f"Error processing {company} ({link}): {e}")
        continue  # Skip to the next link if there's an error
        
    finally:
        # --- Ensure Browser Closes ---
        if driver:
            print(f"Closing browser for {company}.")
            driver.quit()
    
    # Process the HTML to extract Q&A pairs
    try:
        # Parse the saved HTML file instead of using html from driver
        with open(output_path, "r", encoding="utf-8") as file:
            html = file.read()
            
        soup = bs4.BeautifulSoup(html, "html.parser")
        
        # Create a structure for this company
        company_data = {
            "company_id": company,
            "source_html": f"{company}.html",
            "batch": None,  # Will be populated if found
            "status": None, # Will be populated if found
            "description": None, # Will be populated if found
            "qna": []
        }
        
        # First, extract company metadata (batch, status, description)
        # Looking for div elements that might contain the company info
        company_info_divs = soup.find_all("div", class_="ct-text-block")
        
        # Extract batch information
        for div in company_info_divs:
            if div.text and "Batch:" in div.text:
                batch_span = div.find("span", class_="ct-span")
                if batch_span:
                    company_data["batch"] = batch_span.text.strip()
                    print(f"  Found batch: {company_data['batch']}")
        
        # Extract status information
        for div in company_info_divs:
            if div.text and "Status:" in div.text:
                status_span = div.find("span", class_="ct-span")
                if status_span:
                    company_data["status"] = status_span.text.strip()
                    print(f"  Found status: {company_data['status']}")
        
        # Extract company description (usually the first substantial text block)
        description_divs = soup.find_all("div", class_="ct-text-block")
        for div in description_divs:
            span = div.find("span", class_="ct-span")
            if span and len(span.text) > 20 and "Batch:" not in div.text and "Status:" not in div.text:
                company_data["description"] = span.text.strip()
                print(f"  Found description: {company_data['description'][:30]}...")
                break
                
        # Extract Q&A pairs from each card
        for card in soup.find_all("div", class_="ct-div-block card brand-color"):
            # Find the question element within the current card
            question_div = card.find("div", class_="ct-div-block card-top")
            
            # Find the answer element within the current card
            answer_div = card.find("div", class_="ct-div-block card-bottom")
    
            # Get the text from the question element, if found
            question_text = None
            if question_div:
                question_text = question_div.get_text(strip=True) 
    
            # Get the text from the answer element, if found
            answer_text = None
            if answer_div:
                answer_text = answer_div.get_text(strip=True)
    
            # Add the Q&A pair to the company's data if both exist
            if question_text and answer_text:
                company_data["qna"].append({
                    "question": question_text,
                    "answer": answer_text
                })
                print(f"  Added Q&A: {question_text[:30]}...")
            else:
                print("  Warning: Could not find question or answer for a card.")
        
        # Add the company data to the master list
        all_company_data.append(company_data)
        print(f"Added {len(company_data['qna'])} Q&A pairs for {company}")
        
    except Exception as e:
        print(f"Error processing HTML for {company}: {e}")

# Save the collected data as JSON
try:
    output_json_path = os.path.join(processed_data_dir, "company_qna_data.json")
    with open(output_json_path, "w", encoding="utf-8") as json_file:
        json.dump(all_company_data, json_file, indent=2)
    print(f"Successfully saved data for {len(all_company_data)} companies to {output_json_path}")
except Exception as e:
    print(f"Error saving JSON data: {e}")

Processing: dropbox (https://getintoyc.com/company/dropbox/)
Waiting for text 'Batch' using XPath: //*[contains(text(), 'Batch')]
Text 'Batch' found.
Extracting page source...
Saving HTML to: companies_htmls/dropbox.html
Closing browser for dropbox.
  Found batch: 2007 Summer
  Found batch: 2007 Summer
  Found status: Successful
  Found status: Successful
  Found description: A cloud storage service that l...
  Added Q&A: If you have a demo, what's the...
  Added Q&A: What is your company going to ...
  Added Q&A: Please tell us about an intere...
  Added Q&A: Please tell us in one or two s...
  Added Q&A: How long have the founders kno...
  Added Q&A: If we fund you, which of the f...
  Added Q&A: Do any founders have other com...
  Added Q&A: Do any founders have commitmen...
  Added Q&A: How far along are you?...
  Added Q&A: How long have each of you been...
  Added Q&A: What's new about what you're m...
  Added Q&A: Who are your competitors, and ...
  Added Q&A: What do you unders